## Setting up

In [1]:
# libraries
import os
import tempfile

import scanpy as sc
import scvi
import seaborn as sns
import torch

from pathlib import Path

/Users/robin.carvajal/miniforge3/envs/py-cosmx/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# setting paths
main_dir = Path('/Volumes/robin_work/cosmx_gray')
os.chdir(main_dir)

Load the unintegrated object

In [4]:
# load comb object
comb = sc.read_h5ad('data/comb/h5ad/comb_v3.h5ad')

## Plotting Functions

In [5]:
from plotnine import *
import pandas as pd

In [8]:
# Turn the embedding into a DataFrame and keep cell names as index
df = pd.DataFrame(
    comb.obsm['X_umap'],
    columns=['umap1', 'umap2'],
    index=comb.obs_names  # keep cell names!
)

df['sample_name'] = comb.obs['sample_name']
df['leiden_scVI'] = comb.obs['leiden_scVI']


In [9]:
df

,umap1,umap2,sample_name,leiden_scVI
c_1_100_100,10.338116,2.288259,sample1,0
c_1_100_1002,4.179844,-0.977051,sample1,8
c_1_100_1003,10.672911,5.443767,sample1,2
c_1_100_1004,1.383109,-4.078279,sample1,8
c_1_100_1005,10.526304,1.797765,sample1,0
...,...,...,...,...
c_2_321_1556,15.064250,1.373836,sample5,6
c_2_298_1163,15.679472,1.096530,sample5,6
c_2_307_1460,12.829216,0.975668,sample5,6
c_2_314_141,14.536077,0.857352,sample5,6


In [ ]:
def plt_umap(df, x_col, y_col, group_by, split_by, pt_size):

    p = (
        ggplot(df, aes(x=x_col, y=y_col, color=group_by))
        + geom_point(shape='.', size=pt_size)
        + theme(figure_size=(12,12))
    )

    if split_by:
        p += facet_wrap(facets=split_by)

    return p


In [ ]:
from plotnine import (
    ggplot, aes, geom_point, geom_label, facet_wrap, theme
)
import pandas as pd

def plt_umap(df, x_col, y_col, group_by, split_by=None, pt_size=0.1, add_labels=True):
    """
    Make a UMAP plot with optional facetting and Seurat-style group labels.
    """

    p = (
        ggplot(df, aes(x='umap1', y='umap2', color=group_by))
        + geom_point(shape='.', size=pt_size)
        + theme(figure_size=(12, 12))
    )

    if split_by:
        p += facet_wrap(f'~{split_by}')

    if add_labels:
        centroids = (
            df.groupby(group_by)[['umap1', 'umap2']]
            .median()
            .reset_index()
        )

        # use both color and fill mapped to group_by
        p += geom_label(
            centroids,
            aes(x='umap1', y='umap2', label=group_by, fill=group_by),
            color="black",      # text color
            size=8,
            ha="center"
        )

    return p


In [30]:
p = plt_umap(df, group_by='leiden_scVI', split_by=None, pt_size=0.1, add_labels=True)
q = plt_umap(df, group_by='leiden_scVI', split_by='leiden_scVI', pt_size=0.1, add_labels=False)

/var/folders/16/1r_792_j7w9_sfl4y3w5k1v80000gp/T/ipykernel_78650/1159403196.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


In [31]:
p.save('umap_plt', width=12, height=12, dpi=300)
q.save('umap_plt_split', width=12, height=12, dpi=300)

/Users/robin.carvajal/miniforge3/envs/py-cosmx/lib/python3.10/site-packages/plotnine/ggplot.py:630: PlotnineWarning: Saving 12 x 12 in image.
/Users/robin.carvajal/miniforge3/envs/py-cosmx/lib/python3.10/site-packages/plotnine/ggplot.py:631: PlotnineWarning: Filename: umap_plt
/Users/robin.carvajal/miniforge3/envs/py-cosmx/lib/python3.10/site-packages/plotnine/ggplot.py:630: PlotnineWarning: Saving 12 x 12 in image.
/Users/robin.carvajal/miniforge3/envs/py-cosmx/lib/python3.10/site-packages/plotnine/ggplot.py:631: PlotnineWarning: Filename: umap_plt_split
